# Prueba para hallar el mejor modelo

In [ ]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
import itertools
import numpy as np
from sklearn.model_selection import train_test_split

# ============================
# 1. Cargar datos
# ============================
with open("datos_procesados.json", "r") as f:
    data = json.load(f)

X = np.array([[d["x"], d["y"]] for d in data], dtype=np.float32)
y = np.array([d["label"] for d in data], dtype=np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, shuffle=True
)

X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train)
X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test)

# ============================
# 2. Crear red con arquitectura variable
# ============================
def build_network(neuron_list):
    layers = []
    input_dim = 2

    for n in neuron_list:
        layers.append(nn.Linear(input_dim, n))
        layers.append(nn.ReLU())
        input_dim = n

    layers.append(nn.Linear(input_dim, 1))  # salida
    return nn.Sequential(*layers)

# ============================
# 3. Entrenar red
# ============================
def train_model(model, lr):
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(1000):
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        test_pred = model(X_test)
        test_loss = criterion(test_pred, y_test).item()

    return loss.item(), test_loss

# ============================
# 4. Búsqueda de mejor arquitectura
# ============================
learning_rates = [0.003, 0.01, 0.03, 0.1, 0.3]

best = {
    "test_loss": float("inf"),
    "architecture": None,
    "lr": None,
    "train_loss": None,
    "weights": None
}

# Todas las combinaciones de neuronas por capa
for num_layers in range(1, 7):  # 1 a 6 capas
    for neuron_list in itertools.product(range(1, 9), repeat=num_layers):
        for lr in learning_rates:

            model = build_network(neuron_list)
            train_loss, test_loss = train_model(model, lr)

            print(f"Capas={num_layers}, Arquitectura={neuron_list}, LR={lr} "
                  f"| Train={train_loss:.4f} | Test={test_loss:.4f}")

            if test_loss < best["test_loss"]:
                best["test_loss"] = test_loss
                best["architecture"] = neuron_list
                best["lr"] = lr
                best["train_loss"] = train_loss
                best["weights"] = [p.detach().numpy() for p in model.parameters()]

# ============================
# 5. Imprimir mejor solución
# ============================
print("\n==============================")
print("MEJOR CONFIGURACIÓN ENCONTRADA")
print("==============================")
print(f"Arquitectura (neuronas por capa): {best['architecture']}")
print(f"Learning rate: {best['lr']}")
print(f"Training loss: {best['train_loss']:.4f}")
print(f"Test loss: {best['test_loss']:.4f}")

print("\nPesos y Biases finales:")
for i, w in enumerate(best["weights"]):
    print(f"\nParámetro {i}:\n{w}")


Capas=1, Arquitectura=(1,), LR=0.003 | Train=0.9006 | Test=0.9861
Capas=1, Arquitectura=(1,), LR=0.01 | Train=0.9036 | Test=0.9858
Capas=1, Arquitectura=(1,), LR=0.03 | Train=0.9032 | Test=0.9822
Capas=1, Arquitectura=(1,), LR=0.1 | Train=0.8602 | Test=0.9842
Capas=1, Arquitectura=(1,), LR=0.3 | Train=0.8459 | Test=0.9999
Capas=1, Arquitectura=(2,), LR=0.003 | Train=0.8688 | Test=0.9924
Capas=1, Arquitectura=(2,), LR=0.01 | Train=0.8466 | Test=0.9933
Capas=1, Arquitectura=(2,), LR=0.03 | Train=0.8572 | Test=1.0041
Capas=1, Arquitectura=(2,), LR=0.1 | Train=0.8361 | Test=1.0247
